# Phase 1: data from the verified source, three test sets, and the complaint trends

Runs on a free Colab **CPU** runtime; no GPU and no token are needed. At the end it
downloads `phase1_results.zip`.

1. **Raw data.** `scripts/fetch_booking_515k.py` downloads the Booking 515K CSV
   from a pinned Hugging Face revision and refuses it unless its size and SHA-256
   match the Kaggle file the project was built on.
2. **Three splits of the same reviews** (`data/processed_v2*`): random, out-of-time
   (train on older reviews, test on newer) and unseen hotels.
3. **What each test measures.** `scripts/split_views.py` trains the classical
   candidates, picks the best on dev, and reports test macro-F1 with a bootstrap
   95% interval for each split.
4. **Which complaints are rising.** `scripts/complaint_trends.py` builds a SQLite
   database, runs the SQL in `analysis/complaint_trends/sql/`, compares the same
   months one year apart within the same hotels, and samples quotes for every
   topic it flags. It runs twice: over the full data (September 2015 to July 2017),
   and over the months after February 2016, when the share of reviews with any
   complaint jumped from 61% to 67% in one month (a change in Booking's review
   form lifts every topic at once).

In [ ]:
# True runs everything (about 35 minutes). False redoes only the comparison
# after February 2016 (about 12 minutes), when the rest has already been run.
RUN_EVERYTHING = True

In [ ]:
!git clone --quiet --branch claude/funny-rubin-31xgtb https://github.com/krimits/hotel-review-nlp.git
%cd hotel-review-nlp
!git log --oneline -1
!pip install --quiet --no-deps -e .

In [ ]:
import subprocess


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


run("python scripts/fetch_booking_515k.py")

## 2-3. The three splits and the dev-selected classical model on each

In [ ]:
if RUN_EVERYTHING:
    for config in ("configs/baselines.yaml", "configs/baselines_time.yaml", "configs/baselines_hotel.yaml"):
        run(f"python -m reviewnlp.data.preprocess --config {config}")
    run("python scripts/split_views.py")

## 4. Complaint trends: SQL, within-hotel changes, charts and quote samples

In [ ]:
if RUN_EVERYTHING:
    run("python scripts/complaint_trends.py --workers 2 --out runs/complaint_trends")
run("python scripts/complaint_trends.py --workers 2 --since 2016-02 --out runs/complaint_trends_since_2016_02")

In [ ]:
from IPython.display import Image, display

display(Image("runs/complaint_trends_since_2016_02/within_hotel_change.png"))
display(Image("runs/complaint_trends_since_2016_02/monthly_rates.png"))

## 5. Download the results (no review database: results, manifests, charts and quote samples)

In [ ]:
import glob
import zipfile

from google.colab import files

patterns = [
    "docs/experiments/results/split_views.json",
    "data/processed_v2*/data_manifest.json",
    "data/processed_v2*/label_stats.json",
    "runs/classical_v2*/metrics.json",
    "runs/complaint_trends*/results.json",
    "runs/complaint_trends*/quotes_for_review.json",
    "runs/complaint_trends*/*.png",
    "runs/complaint_trends*/csv/*.csv",
]
paths = sorted({path for pattern in patterns for path in glob.glob(pattern)})
with zipfile.ZipFile("phase1_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path)
print(f"{len(paths)} files")
files.download("phase1_results.zip")